# Evaluating the fine-tuned satellite anomaly report model

This notebook tests if the fine-tuned model  is better than the alternatives, and where it breaks.

Four "arms" are compared on test segments the model never saw (the validation set picked the checkpoint, the test set was never touched):

| Arm | What it is |
|---|---|
| `teacher` | The Sonnet 5 reports from `test.jsonl`. The ceiling, and a calibration reference for every metric. |
| `finetuned` | Your Qwen3-4B + LoRA adapter, same prompt it was trained on. |
| `base_prompted` | The same Qwen3-4B with **no** adapter but a detailed style-rules system prompt. This is the baseline that matters: if prompting alone matches your fine-tune, the fine-tune did not earn its keep. |
| `base` | The same Qwen3-4B, no adapter, the training system prompt. Shows what the adapter changed. |

What gets measured:

1. **Format compliance**: no markdown, no em dashes, no "dataset/median" language, not cut off.
2. **Number faithfulness**: every number in a report is checked against the numbers in its prompt. Run on the teacher first, so you know what a "normal" miss rate looks like.
3. **Verdict agreement**: does the report conclude "no action needed" only for NOMINAL segments?
4. **Timing-cause anchoring**: your teacher prompt leaned on the sampling-gap feature. Does the student over-blame timing faults?
5. **Detector mistakes flowing into reports**: what the model writes when the Random Forest hands it the wrong label.
6. **Blind pairwise LLM judge**: fine-tuned vs prompted base, and fine-tuned vs teacher.

**Workflow:** keep `DRY_RUN = True` first (about 10 minutes, tiny sample) to prove every cell works, then set it to `False` and rerun from the top. Generations are cached to Drive after every batch, so a disconnect only costs you the current batch.

Needs a GPU runtime. Everything except the generation cell and the judge also runs without one.

In [1]:
# pip installs
!pip install -q -U transformers peft bitsandbytes accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/satellite-anomaly-project/')
FT_DIR = PROJECT_DIR / 'finetune'
DATASET_CSV = PROJECT_DIR / 'dataset.csv'
TEST_JSONL = FT_DIR / 'test.jsonl'
TRAIN_JSONL = FT_DIR / 'train.jsonl'
ADAPTER_DIR = FT_DIR / 'adapter_final'
EVAL_DIR = FT_DIR / 'eval'
EVAL_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'

DRY_RUN = False                  # True = tiny sample, separate cache files. Set False for the real evaluation.
PER_CLASS = 4 if DRY_RUN else 75 # anomalies and nominals sampled from the test set (balanced on purpose)
BATCH = 8                        # generation batch size; lower to 4 if you hit out-of-memory
MAX_NEW = 450                    # max new tokens per report (teacher reports are about 230 tokens)
SEED = 42

GEN_FILE = EVAL_DIR / ('generations_dry.jsonl' if DRY_RUN else 'generations.jsonl')

for p in (DATASET_CSV, TEST_JSONL, TRAIN_JSONL, ADAPTER_DIR):
    assert p.exists(), f'Missing {p}. Fix the paths above.'
print('DRY_RUN =', DRY_RUN, '| generations cache:', GEN_FILE.name)

Mounted at /content/drive
DRY_RUN = False | generations cache: generations.jsonl


## 1. Load the test set and rebuild the Random Forest detector

The Random Forest is refit here so we know which test segments the detector gets wrong. The original run reported about 0.95 precision and 0.83 recall. The fitted detector is saved to Drive as `rf_detector.joblib`.

In [3]:
import json, re, math, random, contextlib, time
from collections import Counter
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATASET_CSV)
df['segment'] = df['segment'].astype(int)

FEATURES = ['len', 'duration', 'mean', 'std', 'var', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks',
            'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared']

train_pool = df[df['train'] == 1]
test_all = df[df['train'] == 0].copy()

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(train_pool[FEATURES], train_pool['anomaly'])
test_all['rf_pred'] = rf.predict(test_all[FEATURES])
test_all['rf_proba'] = rf.predict_proba(test_all[FEATURES])[:, 1]

y, p = test_all['anomaly'], test_all['rf_pred']
print(f'Random Forest on all {len(test_all)} test rows: accuracy {accuracy_score(y, p):.3f}, '
      f'precision {precision_score(y, p):.3f}, recall {recall_score(y, p):.3f}')
joblib.dump({'model': rf, 'features': FEATURES}, PROJECT_DIR / 'rf_detector.joblib')

# test.jsonl holds only the segments that survived the quality gate
test_recs = [json.loads(l) for l in open(TEST_JSONL, encoding='utf-8') if l.strip()]
ev = pd.DataFrame({
    'segment': [r['segment'] for r in test_recs],
    'anomaly': [r['anomaly'] for r in test_recs],
    'system': [r['prompt'][0]['content'] for r in test_recs],
    'user': [r['prompt'][1]['content'] for r in test_recs],
    'teacher': [r['completion'][0]['content'] for r in test_recs],
}).merge(test_all[['segment', 'rf_pred', 'rf_proba', 'gaps_squared', 'duration']], on='segment', how='left')
assert ev['rf_pred'].notna().all(), 'Some test segments were not found in dataset.csv'
assert (ev['user'].str.contains('Label: ANOMALY') == (ev['anomaly'] == 1)).all(), 'Label line disagrees with anomaly column'
ev['rf_error'] = ev['rf_pred'] != ev['anomaly']
print(f'test.jsonl: {len(ev)} segments, {int(ev.anomaly.sum())} anomalies, {int(ev.rf_error.sum())} detector errors')

anom, nom = ev[ev.anomaly == 1], ev[ev.anomaly == 0]
picked = pd.concat([anom.sample(min(PER_CLASS, len(anom)), random_state=SEED),
                    nom.sample(min(PER_CLASS, len(nom)), random_state=SEED)])
errors = ev[ev.rf_error]
errors = errors.head(4) if DRY_RUN else errors
eval_df = pd.concat([picked, errors]).drop_duplicates('segment').reset_index(drop=True)
print(f'evaluation set: {len(eval_df)} segments ({int(eval_df.anomaly.sum())} anomalies, {int(eval_df.rf_error.sum())} detector errors)')

Random Forest on all 529 test rows: accuracy 0.964, precision 0.927, recall 0.903
test.jsonl: 520 segments, 109 anomalies, 18 detector errors
evaluation set: 158 segments (79 anomalies, 18 detector errors)


## 2. The metric functions, validated on the teacher reports (free, no GPU)

Each metric is a rule, and rules can be wrong. So before using them on the student, we run them on the 500-odd **teacher** reports in `test.jsonl`. That tells you what a "good" report scores, and exposes any metric that misfires.

- **Number faithfulness.** Every number in the report is compared with the numbers in its prompt. Integers must match exactly; decimals and scientific notation match within 3% (to allow rounding). A report number with no counterpart in the prompt is "unsupported". Some of those are legitimate derived values (a ratio, a sum), so the teacher's rate is the baseline, not zero.
- **Verdict.** A report that says "no action needed", "no corrective action" or "routine/standard monitoring" counts as a NOMINAL verdict. Anything else counts as ANOMALY. Crude, but the training reports are very consistent about this, and the teacher check below prints any mismatches so you can judge whether the rule holds.
- **Timing-focused next step.** The recommendation is the last sentence of each report. It counts as timing-focused when it tells ops to check timestamps, packet logs, buffers, the downlink, or sampling clocks. This is measured on the recommendation only, because "sampling gap irregularity" is a feature name that appears in almost every report, including nominal ones.

In [4]:
NUM_RE = re.compile(r'(?<![\w.])[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?(?!\w)')
EM_RE = re.compile('—')
MD_RE = re.compile(r'\*\*|^\s{0,3}#{1,6}\s|^\s*[-*•]\s|^\s*\d+[.)]\s', re.M)
DS_RE = re.compile(r'percentile|median|dataset', re.I)
NO_ACTION_RE = re.compile(r'no (further |immediate |corrective )?(action|intervention)|routine (monitoring|archival)|'
                          r'standard monitoring|continue (routine|standard|normal|nominal|regular)|label nominal is supported', re.I)
# The sentence that tells ops what to do next. In these reports it is the last sentence.
TIMING_ACTION_RE = re.compile(r'timestamp (log|record)|(onboard|raw) timestamps?|timestamp/packet|packet|downlink|'
                              r'telemetry (buffer|chain|frame|handling)|buffer|bus (health|timing|logs?)|'
                              r'sampling (clock|irregularity)|timing|data[- ]handling|ground[- ]station|clock', re.I)

def action_sentence(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return sents[-1] if sents else ''

def is_int_str(s):
    return re.fullmatch(r'[-+]?\d+', s) is not None

def prompt_numbers(user_msg):
    body = '\n'.join(l for l in user_msg.splitlines() if not l.startswith(('Channel:', 'Label:')))
    return set(NUM_RE.findall(body))

def sig_digits(s):
    mant = re.split(r'[eE]', s.lstrip('+-'))[0]
    return max(len(mant.replace('.', '').lstrip('0')), 1)

def is_supported(num, pnums):
    if num in pnums:
        return True
    x = float(num)
    k = sig_digits(num)
    for q in pnums:
        z = float(q)
        if is_int_str(num) and is_int_str(q):
            if x == z:
                return True
        elif not is_int_str(num):
            # decimals and scientific notation: within 3%, or equal to the prompt value rounded to the
            # report's own number of significant digits (so "~1e-5" matches 1.17e-05)
            if math.isclose(x, z, rel_tol=0.03) or (z != 0 and float(f'{z:.{k - 1}e}') == x):
                return True
    return False

def score_text(text, user_msg):
    nums = NUM_RE.findall(text)
    pn = prompt_numbers(user_msg)
    bad = [n for n in nums if not is_supported(n, pn)]
    return dict(
        words=len(text.split()),
        em_dash=bool(EM_RE.search(text)),
        markdown=bool(MD_RE.search(text)),
        dataset_lang=bool(DS_RE.search(text)),
        n_numbers=len(nums),
        n_unsupported=len(bad),
        unsupported=bad,
        verdict=0 if NO_ACTION_RE.search(text) else 1,     # 1 = says ANOMALY, 0 = says NOMINAL
        timing_action=bool(TIMING_ACTION_RE.search(action_sentence(text))),
    )

def wilson(k, n, z=1.96):
    if n == 0:
        return float('nan'), float('nan')
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def rate(mask):
    k, n = int(np.sum(mask)), len(mask)
    lo, hi = wilson(k, n)
    return f'{100 * k / n:.0f}% [{100 * lo:.0f}-{100 * hi:.0f}]'

# Validate every metric on the teacher reports, all of test.jsonl
tv = pd.DataFrame([score_text(r.teacher, r.user) for r in ev.itertuples()])
tv['anomaly'] = ev['anomaly'].values
print(f'TEACHER REPORTS (n={len(tv)}), 95% intervals in brackets')
print('  em dash:                ', rate(tv.em_dash))
print('  markdown:               ', rate(tv.markdown))
print('  dataset/median language:', rate(tv.dataset_lang))
print('  any unsupported number: ', rate(tv.n_unsupported > 0), f'| mean numbers cited per report: {tv.n_numbers.mean():.1f}')
print('  verdict matches label:  ', rate(tv.verdict == tv.anomaly))
print('\nverdict vs label (rows = true label, columns = verdict in the report):')
print(pd.crosstab(tv.anomaly, tv.verdict, rownames=['label'], colnames=['verdict']))
miss = Counter(n for lst in tv.unsupported for n in lst)
print('\nmost common "unsupported" numbers in teacher reports (usually derived values or coarse rounding):', miss.most_common(8))
bad_v = ev[(tv.verdict != tv.anomaly).values].head(3)
for r in bad_v.itertuples():
    print(f'\n--- verdict mismatch, segment {r.segment}, label {r.anomaly} ---\n{r.teacher[-300:]}')

TEACHER REPORTS (n=520), 95% intervals in brackets
  em dash:                 0% [0-1]
  markdown:                0% [0-1]
  dataset/median language: 0% [0-1]
  any unsupported number:  8% [6-11] | mean numbers cited per report: 6.5
  verdict matches label:   99% [97-99]

verdict vs label (rows = true label, columns = verdict in the report):
verdict    0    1
label            
0        404    7
1          0  109

most common "unsupported" numbers in teacher reports (usually derived values or coarse rounding): [('1e-13', 14), ('1e-12', 11), ('1e-10', 7), ('1e-6', 5), ('1e-5', 4), ('1e-11', 4), ('7', 2), ('1e-4', 2)]

--- verdict mismatch, segment 229, label 0 ---
 but bounded physical event; recommend a quick cross-check against related channels or subsystem state at this timestamp to confirm the excursion aligns with an expected operational mode, but no immediate corrective action is indicated given the NOMINAL label and lack of erratic derivative behavior.

--- verdict mismatch, segme

## 3. Load the base model and the adapter

One copy of the 4-bit base model is loaded with the LoRA adapter attached. The `base` and `base_prompted` arms run the **same weights** with the adapter switched off (`disable_adapter()`), so the comparison is exact and uses no extra memory.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > pick a GPU.'
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

tok = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
tok.padding_side = 'left'            # required for batched generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
                                            device_map={'': 0}, dtype=compute_dtype)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('loaded. GPU memory used (GB):', round(torch.cuda.memory_allocated() / 1e9, 1))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loaded. GPU memory used (GB): 2.8


In [6]:
def generate_batch(messages_list, adapter_on):
    """Greedy generation for a batch of chat-message lists. Returns (texts, truncated_flags)."""
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list]
    enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
    ctx = contextlib.nullcontext() if adapter_on else model.disable_adapter()
    with torch.no_grad(), ctx:
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id, use_cache=True)
    gen = out[:, enc['input_ids'].shape[1]:]
    decoded = [tok.decode(g, skip_special_tokens=True).strip() for g in gen]
    truncated = [int((g != tok.pad_token_id).sum()) >= MAX_NEW for g in gen]
    return decoded, truncated

# Quick smoke test on one segment
r0 = eval_df.iloc[0]
t0 = time.time()
out, _ = generate_batch([[{'role': 'system', 'content': r0['system']}, {'role': 'user', 'content': r0['user']}]], adapter_on=True)
print(f'{time.time() - t0:.0f}s for one report\n')
print(out[0])

41s for one report

This segment stands out mainly for its elevated skew (1.205) combined with a moderate but not extreme mean and variance, and a sampling gap irregularity of 850, which is well above what's seen in a normally sampled channel. The raw signal shows only 2 peaks, which collapses to a single peak once smoothed, indicating the shape is driven by a brief, one-sided excursion rather than repeated oscillation or noise. The second-derivative variance (0.00876) is notably higher than the first-derivative variance (0.00311), suggesting the signal changes abruptly at the point of the excursion, consistent with a step-like or spike event rather than a smooth trend. Given the low peak count and high second-derivative activity, this looks like a single transient event (e.g., a sensor glitch, valve firing, or brief thermal/power transient) rather than a sustained anomaly or sensor noise floor issue. Recommend pulling the raw waveform around this window to confirm the event's shape an

## 4. Generate reports for every arm

- **`label_mode = 'true'`:** the prompt carries the real label, exactly like training.
- **`label_mode = 'rf'`:** for the segments the Random Forest got wrong only, the `Label:` line is replaced with the detector's (wrong) flag. This is what the deployed pipeline would do.

Every batch is appended to a file on Drive. If Colab disconnects, restart, rerun the cells above. This cell resumes where it stopped.

In [7]:
STYLE_SYSTEM = (
    'You are a spacecraft operations engineer. You receive pre-computed statistical features of one flagged '
    'telemetry segment (not raw sensor values) plus its label. Write a concise mission-ops assessment as one '
    'paragraph of plain prose, about 120 to 200 words. Rules: no markdown, no bullet points, no headings, no em '
    'dashes. Cite the specific statistics that drive your assessment, with their values exactly as given, and '
    'never state a value that is not in the input. Do not mention percentiles, medians, datasets or typical '
    'values, because you only see this one segment. State the likely cause and finish with one concrete next '
    'step for the ops team. If the label is NOMINAL, explain briefly why the segment looks healthy and say that '
    'no action is needed.'
)

def build_messages(arm, row, label_mode):
    user = row['user']
    if label_mode == 'rf':
        lab = 'ANOMALY' if row['rf_pred'] == 1 else 'NOMINAL'
        new_user = re.sub(r'Label: (ANOMALY|NOMINAL)\s*$', f'Label: {lab}', user)
        assert new_user != user, 'label swap did not change the prompt'
        user = new_user
    system = STYLE_SYSTEM if arm == 'base_prompted' else row['system']
    return [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]

def job_key(seg, arm, mode):
    return f'{seg}|{arm}|{mode}'

done = {}
if GEN_FILE.exists():
    for line in open(GEN_FILE, encoding='utf-8'):
        if line.strip():
            d = json.loads(line)
            done[d['key']] = d

rows = eval_df.to_dict('records')
plan = [('finetuned', 'true', rows),
        ('finetuned', 'rf', [r for r in rows if r['rf_error']]),
        ('base_prompted', 'true', rows)]
        # ('base', 'true', rows)]

t_start = time.time()
for arm, mode, group in plan:
    todo = [r for r in group if job_key(r['segment'], arm, mode) not in done]
    todo.sort(key=lambda r: len(r['user']))          # similar lengths per batch = less padding
    print(f'[{arm} / {mode}] {len(group) - len(todo)} cached, {len(todo)} to generate')
    for i in range(0, len(todo), BATCH):
        chunk = todo[i:i + BATCH]
        texts, trunc = generate_batch([build_messages(arm, r, mode) for r in chunk], adapter_on=(arm == 'finetuned'))
        with open(GEN_FILE, 'a', encoding='utf-8') as f:
            for r, t, tr in zip(chunk, texts, trunc):
                rec = {'key': job_key(r['segment'], arm, mode), 'segment': int(r['segment']), 'arm': arm,
                       'label_mode': mode, 'text': t, 'truncated': bool(tr)}
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
                done[rec['key']] = rec
        print(f'  {min(i + BATCH, len(todo))}/{len(todo)} done, {(time.time() - t_start) / 60:.1f} min elapsed')
print('all generations complete:', len(done))

[finetuned / true] 158 cached, 0 to generate
[finetuned / rf] 18 cached, 0 to generate
[base_prompted / true] 158 cached, 0 to generate
all generations complete: 334


## 5. Score everything and compare the arms

Read this table with the intervals in mind: with about 150 reports per arm, a rate near 50% is only pinned down to roughly plus or minus 8 points. Differences smaller than the interval widths are noise.

**How to read it**

- `format_ok` is where the adapter should clearly beat `base`. The honest test is `base_prompted`: a strong prompt may close most of the format gap.
- `any unsupported number` for `finetuned` should sit close to the `teacher` row. If it is much higher, the student is inventing values.
- `verdict` rows show whether the report agrees with the label it was given. Anything well below the teacher means the model is ignoring the label or hedging.

In [8]:
gens = pd.DataFrame(list(done.values()))
keep_cols = ['segment', 'anomaly', 'user', 'teacher', 'gaps_squared', 'duration', 'rf_pred', 'rf_proba', 'rf_error']
g = gens[gens.segment.isin(eval_df.segment)].merge(eval_df[keep_cols], on='segment')
teach = eval_df[keep_cols].assign(arm='teacher', label_mode='true', text=eval_df['teacher'], truncated=False, key='')
long = pd.concat([g, teach], ignore_index=True)

sc = pd.DataFrame([score_text(r.text, r.user) for r in long.itertuples()])
scored = pd.concat([long.reset_index(drop=True), sc], axis=1)
scored['format_ok'] = ~(scored.em_dash | scored.markdown | scored.dataset_lang | scored.truncated)

ARM_ORDER = ['teacher', 'finetuned', 'base_prompted', 'base']
main = scored[scored.label_mode == 'true']
rows_out = []
for arm in ARM_ORDER:
    a = main[main.arm == arm]
    if a.empty:
        continue
    rows_out.append({
        'arm': arm, 'n': len(a),
        'format_ok': rate(a.format_ok),
        '  em dash': rate(a.em_dash), '  markdown': rate(a.markdown),
        '  dataset language': rate(a.dataset_lang), '  truncated': rate(a.truncated),
        'mean words': round(a.words.mean()),
        'numbers cited / report': round(a.n_numbers.mean(), 1),
        'any unsupported number (lower is better)': rate(a.n_unsupported > 0),
        'verdict: ANOMALY on true anomalies': rate(a[a.anomaly == 1].verdict == 1),
        'verdict: NOMINAL on true nominals': rate(a[a.anomaly == 0].verdict == 0),
    })
summary = pd.DataFrame(rows_out).set_index('arm')
display(summary.T)
summary.to_csv(EVAL_DIR / ('summary_dry.csv' if DRY_RUN else 'summary.csv'))

arm,teacher,finetuned,base_prompted
n,158,158,158
format_ok,100% [98-100],100% [98-100],41% [33-48]
em dash,0% [0-2],0% [0-2],59% [52-67]
markdown,0% [0-2],0% [0-2],0% [0-2]
dataset language,0% [0-2],0% [0-2],0% [0-2]
truncated,0% [0-2],0% [0-2],0% [0-2]
mean words,164,158,156
numbers cited / report,5.7,6.4,10.7
any unsupported number (lower is better),6% [3-10],6% [3-10],1% [0-4]
verdict: ANOMALY on true anomalies,100% [95-100],100% [95-100],100% [95-100]


## 6. Did the student inherit the "timing fault" story?

The distillation teacher model prompt leaned on `gaps_squared`, and 234 rows in the dataset have `gaps_squared == duration`. In the sample reports that were checked, about 12 of 13 anomaly reports ended by telling ops to check timestamps, packet logs, buffers or the downlink. If the teacher did that regardless of the evidence, the student learned the same shortcut.

This table splits the anomaly reports by whether the gap feature actually equals the segment duration, and shows how often each arm's recommended next step is timing-focused. It is descriptive, not pass/fail. The thing to look for is a high rate in the `gaps_squared != duration` group, where the evidence for a timing fault is weaker. If the training-set rate is already high, the bias is in the data, and the student is just faithful to it.

In [9]:
def gap_group(d):
    return np.where(d['gaps_squared'] == d['duration'], 'gaps_squared == duration', 'gaps_squared != duration')

a = main[main.anomaly == 1].copy()
a['group'] = gap_group(a)
tab = a.groupby(['group', 'arm']).agg(n=('timing_action', 'size'), timing_action=('timing_action', 'mean')).reset_index()
tab['timing_action'] = (100 * tab['timing_action']).round(0).astype(int).astype(str) + '%'
display(tab.pivot(index='arm', columns='group', values=['n', 'timing_action']).reindex(ARM_ORDER))

# The same measurement on the training reports shows whether the bias came from the data
tr = pd.DataFrame([json.loads(l) for l in open(TRAIN_JSONL, encoding='utf-8') if l.strip()])
tr = tr[tr.anomaly == 1].copy()
tr['report'] = tr['completion'].apply(lambda c: c[0]['content'])
tr = tr.merge(df[['segment', 'gaps_squared', 'duration']], on='segment')
tr['timing_action'] = tr.report.apply(lambda t: bool(TIMING_ACTION_RE.search(action_sentence(t))))
tr['group'] = gap_group(tr)
print('\nTraining-set teacher reports on anomalies (what the student learned from):')
print(tr.groupby('group').timing_action.agg(n='size', share='mean').round(2))

n                           \
group         gaps_squared != duration gaps_squared == duration   
arm                                                               
teacher                             68                       11   
finetuned                           68                       11   
base_prompted                       68                       11   
base                               NaN                      NaN   

                         timing_action                           
group         gaps_squared != duration gaps_squared == duration  
arm                                                              
teacher                            63%                      82%  
finetuned                          82%                      82%  
base_prompted                      90%                       9%  
base                               NaN                      NaN


Training-set teacher reports on anomalies (what the student learned from):
                            n  share
group                               
gaps_squared != duration  205   0.70
gaps_squared == duration   57   0.61


## 7. What happens when the detector is wrong?

This is the limitation that matters for the real deployed version. The student was trained with true labels. At deployment its label comes from the Random Forest, which misses some anomalies and raises some false alarms. Here, for exactly those segments, the `Label:` line carries the detector's wrong flag.

If the report simply agrees with the detector, the pipeline is only ever as good as the detector, and a missed anomaly becomes a confident "no action needed".

In [10]:
e = scored[(scored.arm == 'finetuned') & (scored.label_mode == 'rf')].copy()
if e.empty:
    print('No detector-error generations yet.')
else:
    e['error_type'] = np.where(e.anomaly == 1, 'missed anomaly (detector said NOMINAL)', 'false alarm (detector said ANOMALY)')
    e['follows_detector'] = e.verdict == e.rf_pred
    e['follows_truth'] = e.verdict == e.anomaly
    out = e.groupby('error_type').agg(n=('segment', 'size'), report_follows_detector=('follows_detector', 'mean'),
                                      mean_detector_proba=('rf_proba', 'mean')).round(2)
    display(out)
    print('Overall, the report followed the detector on', f'{100 * e.follows_detector.mean():.0f}% of detector errors.')
    print('\nExamples to read (the label line was the detector\'s):')
    for et, grp in e.groupby('error_type'):
        for r in grp.head(2).itertuples():
            print(f'\n=== {et} | segment {r.segment} | detector probability of anomaly {r.rf_proba:.2f} ===')
            print(r.text)

,n,report_follows_detector,mean_detector_proba
error_type,,,
false alarm (detector said ANOMALY),8,1.0,0.64
missed anomaly (detector said NOMINAL),10,1.0,0.20


Overall, the report followed the detector on 100% of detector errors.

Examples to read (the label line was the detector's):

=== false alarm (detector said ANOMALY) | segment 1522 | detector probability of anomaly 0.62 ===
This segment stands out mainly for its elevated mean and variance combined with a single dominant peak that persists across both raw and smoothed traces (1/1/1), meaning the signal isn't showing noisy multi-modal behavior but rather a sustained elevated level with a single broad excursion. The positive skew (0.747) with near-zero kurtosis (-1.104) suggests a broad, lopsided rise rather than a sharp spike or oscillation, consistent with a slow ramp or plateau rather than a transient glitch. The second-derivative peak count (8) against a low first-derivative peak count (2) indicates the signal is changing direction multiple times over its duration, but the overall shape is still dominated by one broad feature, not high-frequency chatter. The sampling gap irregularity 

## 8. Blind pairwise LLM judge

A stronger model reads two reports for the same segment, in random order, without knowing which is which. It returns a winner and counts the claims in each report that the features do not support.

Be aware of two biases, and say so in your write-up:
- The judge (Sonnet 5) is the same model that wrote the teacher reports, so it may favour the teacher's phrasing. Treat `finetuned vs teacher` as "how close to the ceiling", not as an unbiased contest.
- Position bias is controlled by randomizing A/B, but 60 comparisons per pair is a small sample. Report win/tie/loss counts, not just a percentage.

**Cost control:** This makes about 120 sequential calls (no threads). Results are cached line by line on Drive, so reruns skip finished comparisons, and `MAX_JUDGE_CALLS` is a hard stop.

In [11]:
import requests
from google.colab import userdata

BACKBOARD_API_KEY = userdata.get('BACKBOARD_API_KEY')
BACKBOARD_API_URL = 'https://app.backboard.io/api'
BACKBOARD_LLM_PROVIDER = 'anthropic'
BACKBOARD_MODEL_NAME = 'claude-sonnet-5'

if not BACKBOARD_API_KEY:
    raise ValueError('Set BACKBOARD_API_KEY in Colab Secrets before running the judge cell.')

JUDGE_PAIRS = [('finetuned', 'base_prompted'), ('finetuned', 'teacher')]
N_JUDGE = 6 if DRY_RUN else 60          # comparisons per pair, split evenly between anomalies and nominals
MAX_JUDGE_CALLS = 20 if DRY_RUN else 140
JUDGE_FILE = EVAL_DIR / ('judge_dry.jsonl' if DRY_RUN else 'judge.jsonl')

def call_judge(prompt: str) -> str:
    """Return the judge model's raw text reply for one prompt string."""
    response = requests.post(
        f'{BACKBOARD_API_URL}/threads/messages',
        headers={'X-API-Key': BACKBOARD_API_KEY},
        json={
            'content': prompt,
            'llm_provider': BACKBOARD_LLM_PROVIDER,
            'model_name': BACKBOARD_MODEL_NAME,
            'stream': False,
            'memory': 'off',
            'system_prompt': 'You are an impartial evaluator of mission-ops anomaly reports. Reply with JSON only.',
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return payload.get('content') or payload.get('message') or ''

JUDGE_TEMPLATE = """You are reviewing two mission-ops anomaly reports written for the same spacecraft telemetry segment. You are given the exact features the writers saw (including the label) and the two reports.

FEATURES:
{features}

REPORT A:
{a}

REPORT B:
{b}

Judge on: (1) faithfulness: every number and claim must be supported by the features; invented values or unsupported physical causes count against a report; (2) sound, specific engineering reasoning for a mission-ops reader; (3) a concrete next step that fits the label; (4) concise plain prose. Do not reward length or any particular writing style.

Reply with only JSON in exactly this form: {{"unsupported_a": <int, number of claims or values in A not supported by the features>, "unsupported_b": <int>, "winner": "A" or "B" or "tie", "reason": "<one sentence>"}}"""

def parse_judge(reply):
    m = re.search(r'\{.*\}', reply, re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    return d if d.get('winner') in ('A', 'B', 'tie') else None

judged = {}
if JUDGE_FILE.exists():
    for line in open(JUDGE_FILE, encoding='utf-8'):
        if line.strip():
            d = json.loads(line)
            judged[d['key']] = d

def text_of(arm, seg):
    return main[(main.arm == arm) & (main.segment == seg)].text.iloc[0]

ids_a = eval_df[eval_df.anomaly == 1].segment.tolist()[:N_JUDGE // 2]
ids_n = eval_df[eval_df.anomaly == 0].segment.tolist()[:N_JUDGE // 2]
judge_ids = ids_a + ids_n

calls = 0
try:
    for arm_x, arm_y in JUDGE_PAIRS:
        for seg in judge_ids:
            key = f'{seg}|{arm_x}|{arm_y}'
            if key in judged:
                continue
            if calls >= MAX_JUDGE_CALLS:
                raise StopIteration
            flip = random.Random(key).random() < 0.5
            a_arm, b_arm = (arm_y, arm_x) if flip else (arm_x, arm_y)
            feats = eval_df[eval_df.segment == seg].user.iloc[0]
            prompt = JUDGE_TEMPLATE.format(features=feats, a=text_of(a_arm, seg), b=text_of(b_arm, seg))
            parsed = parse_judge(call_judge(prompt))
            calls += 1
            if parsed is None:
                parsed = parse_judge(call_judge(prompt))      # one retry
                calls += 1
            if parsed is None:
                print('could not parse a judge reply for', key)
                continue
            rec = {'key': key, 'segment': seg, 'a_arm': a_arm, 'b_arm': b_arm, 'winner': parsed['winner'],
                   'unsupported_a': parsed.get('unsupported_a'), 'unsupported_b': parsed.get('unsupported_b'),
                   'reason': parsed.get('reason', '')}
            with open(JUDGE_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            judged[key] = rec
            print(f'call {calls}: {key} -> {rec["winner"]}')
except NotImplementedError as ex:
    print(ex)
except StopIteration:
    print(f'Stopped at MAX_JUDGE_CALLS = {MAX_JUDGE_CALLS}. Raise it and rerun to continue (finished pairs are cached).')

if judged:
    jd = pd.DataFrame(list(judged.values()))
    for arm_x, arm_y in JUDGE_PAIRS:
        s = jd[(jd.a_arm.isin([arm_x, arm_y])) & (jd.b_arm.isin([arm_x, arm_y]))]
        if s.empty:
            continue
        win = ((s.winner == 'A') & (s.a_arm == arm_x)) | ((s.winner == 'B') & (s.b_arm == arm_x))
        lose = ((s.winner == 'A') & (s.a_arm == arm_y)) | ((s.winner == 'B') & (s.b_arm == arm_y))
        tie = s.winner == 'tie'
        ux = np.where(s.a_arm == arm_x, s.unsupported_a, s.unsupported_b).astype(float)
        uy = np.where(s.a_arm == arm_x, s.unsupported_b, s.unsupported_a).astype(float)
        print(f'\n{arm_x} vs {arm_y} (n={len(s)}): {int(win.sum())} wins, {int(tie.sum())} ties, {int(lose.sum())} losses '
              f'| score {(win.sum() + 0.5 * tie.sum()) / len(s):.2f} (0.50 = parity)')
        print(f'  judge-counted unsupported claims per report: {arm_x} {np.nanmean(ux):.2f}, {arm_y} {np.nanmean(uy):.2f}')


finetuned vs base_prompted (n=60): 53 wins, 2 ties, 5 losses | score 0.90 (0.50 = parity)
  judge-counted unsupported claims per report: finetuned 0.55, base_prompted 1.47

finetuned vs teacher (n=60): 21 wins, 21 ties, 18 losses | score 0.53 (0.50 = parity)
  judge-counted unsupported claims per report: finetuned 0.40, teacher 0.42


## 9. Read the reports

Final evaluation. Looking for reasoning that contradicts the features, the same cause repeated across unrelated segments, invented physical mechanisms, and endings that trail off.

In [12]:
def show(seg):
    r = eval_df[eval_df.segment == seg].iloc[0]
    print('=' * 90)
    print(r['user'])
    print('\n--- FINE-TUNED ---');    print(text_of('finetuned', seg))
    print('\n--- PROMPTED BASE ---'); print(text_of('base_prompted', seg))
    print('\n--- TEACHER ---');       print(text_of('teacher', seg))

for seg in eval_df.sample(min(5, len(eval_df)), random_state=1).segment:
    show(seg)

print('\n\n######## fine-tuned reports with the most unsupported numbers ########')
worst = main[(main.arm == 'finetuned')].sort_values('n_unsupported', ascending=False).head(3)
for r in worst.itertuples():
    print(f'\nsegment {r.segment}: unsupported numbers {r.unsupported}')
    print(r.text)

Channel: CADC0873
Segment length: 193 samples over 192 time units
Mean: -2.86559e-06, Std dev: 1.36456e-05, Variance: 1.86201e-10
Kurtosis: -1.287, Skew: 0.075
Peak count (raw / smoothed-10 / smoothed-20): 2 / 2 / 2
Derivative peak count (1st / 2nd diff): 1 / 3
Derivative variance (1st / 2nd diff): 4.64175e-12 / 8.99577e-12
Sampling gap irregularity: 192
Label: ANOMALY

--- FINE-TUNED ---
This segment's amplitude statistics (mean, std, variance) are all extremely small and unremarkable, and the distribution shape (kurtosis -1.287, skew 0.075) is flat and symmetric with no sign of spiking or lopsidedness. Peak counts are low and stable across raw and smoothed versions (2/2/2), and derivative variances are tiny, indicating no abrupt jumps or oscillatory behavior in the signal itself. The one clear outlier here is the sampling gap irregularity value of 192, essentially equal to the full segment length, which points to a timing or acquisition problem rather than a physical excursion in the

In [13]:
# Save everything
scored.drop(columns=['unsupported']).to_csv(EVAL_DIR / ('scored_dry.csv' if DRY_RUN else 'scored.csv'), index=False)
try:
    print(summary.T.to_markdown())
except Exception:
    print(summary.T.to_string())
print('\nSaved to', EVAL_DIR)

|                                          | teacher       | finetuned     | base_prompted   |
|:-----------------------------------------|:--------------|:--------------|:----------------|
| n                                        | 158           | 158           | 158             |
| format_ok                                | 100% [98-100] | 100% [98-100] | 41% [33-48]     |
| em dash                                  | 0% [0-2]      | 0% [0-2]      | 59% [52-67]     |
| markdown                                 | 0% [0-2]      | 0% [0-2]      | 0% [0-2]        |
| dataset language                         | 0% [0-2]      | 0% [0-2]      | 0% [0-2]        |
| truncated                                | 0% [0-2]      | 0% [0-2]      | 0% [0-2]        |
| mean words                               | 164           | 158           | 156             |
| numbers cited / report                   | 5.7           | 6.4           | 10.7            |
| any unsupported number (lower is better) | 6% [3

## 10. Additional Testing

In [14]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, joblib
from pathlib import Path
from scipy.stats import binomtest
P = Path('/content/drive/MyDrive/satellite-anomaly-project')

# 1. teacher vs student: timing-focused next step on the same anomaly segments
main = pd.read_csv(P / 'finetune/eval/scored.csv')
main = main[main.label_mode == 'true']
a = main[main.anomaly == 1]
p = a.pivot(index='segment', columns='arm', values='timing_action')[['teacher', 'finetuned']].dropna().astype(bool)
only_student = int((p.finetuned & ~p.teacher).sum())
only_teacher = int((~p.finetuned & p.teacher).sum())
print(f'n={len(p)} | student timing-focused but teacher not: {only_student} | the reverse: {only_teacher}')
print('sign test p-value:', round(binomtest(only_student, only_student + only_teacher, 0.5).pvalue, 3))

# 2. "needs review" band for the demo
d = joblib.load(P / 'rf_detector.joblib')
test_all = pd.read_csv(P / 'dataset.csv')
test_all = test_all[test_all.train == 0].copy()
test_all['rf_proba'] = d['model'].predict_proba(test_all[d['features']])[:, 1]
test_all['rf_pred'] = d['model'].predict(test_all[d['features']])
err = test_all.rf_pred != test_all.anomaly
for lo, hi in [(0.1, 0.9), (0.2, 0.8), (0.3, 0.7)]:
    band = test_all.rf_proba.between(lo, hi)
    print(f'band {lo}-{hi}: {band.mean():.0%} of segments flagged for review, catches {int((band & err).sum())} of {int(err.sum())} detector errors')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
n=79 | student timing-focused but teacher not: 22 | the reverse: 9
sign test p-value: 0.029
band 0.1-0.9: 20% of segments flagged for review, catches 18 of 19 detector errors
band 0.2-0.8: 12% of segments flagged for review, catches 13 of 19 detector errors
band 0.3-0.7: 7% of segments flagged for review, catches 8 of 19 detector errors


11. Export Data for Demo App

In [15]:
import sys, importlib, json
sys.path.insert(0, '/content')
import export_demo_data as E
importlib.reload(E)

DEMO_DIR = FT_DIR / 'demo_data'
demo_cache_file = EVAL_DIR / 'demo_generations.jsonl'
cache = dict(done)                                   # reports the evaluation already generated
if demo_cache_file.exists():                         # resume after a disconnect
    for line in open(demo_cache_file, encoding='utf-8'):
        if line.strip():
            rec = json.loads(line); cache[rec['key']] = rec

segments = E.build_demo_data(
    test_all=test_all, ev=ev, scored=scored, judged=judged, cache=cache,
    generate_fn=lambda msgs: generate_batch(msgs, adapter_on=True)[0],
    out_dir=DEMO_DIR, system_prompt=ev['system'].iloc[0], cache_path=demo_cache_file)

prompt parity check passed on 520 segments
334 reports cached, 372 to generate
  generated 8/372
  generated 16/372
  generated 24/372
  generated 32/372
  generated 40/372
  generated 48/372
  generated 56/372
  generated 64/372
  generated 72/372
  generated 80/372
  generated 88/372
  generated 96/372
  generated 104/372
  generated 112/372
  generated 120/372
  generated 128/372
  generated 136/372
  generated 144/372
  generated 152/372
  generated 160/372
  generated 168/372
  generated 176/372
  generated 184/372
  generated 192/372
  generated 200/372
  generated 208/372
  generated 216/372
  generated 224/372
  generated 232/372
  generated 240/372
  generated 248/372
  generated 256/372
  generated 264/372
  generated 272/372
  generated 280/372
  generated 288/372
  generated 296/372
  generated 304/372
  generated 312/372
  generated 320/372
  generated 328/372
  generated 336/372
  generated 344/372
  generated 352/372
  generated 360/372
  generated 368/372
  generated 37